In [28]:
from flowrec.data.datamodule import AmazonReviewDataConfig, AmazonReviewDataModule
from flowrec.item_representation.content_embeddings import (
    AmazonItemTextBuilder,
    ContentEmbeddingBuilder,
    HashingTextEmbedder,
)
from pathlib import Path

from flowrec.item_representation.content_embeddings import ItemEmbeddingStore
from flowrec.item_representation.semantic_id.rqvae import (
    RQVAEConfig,
    RQVAETrainConfig)
from flowrec.item_representation.semantic_id.rqvae_tokenizer import  RQVAEItemTokenizer, RQVAEItemTokenizerConfig

In [29]:
cfg = AmazonReviewDataConfig(
    reviews_path="../data/raw/2018/Musical_Instruments.json.gz",
    metadata_path="../data/raw/2018/meta_Musical_Instruments.json.gz",
    item_key="asin",

    min_user_interactions=5,
    min_item_interactions=5,

    split_strategy="global_time",
    global_val_ratio=0.1,
    global_test_ratio=0.1,
    global_history_policy="rolling",
    global_eval_target_mode="last_per_user",
    deduplicate_consecutive = True,

    max_history_len=20,
    batch_size=256,
    eval_batch_size=512,
)

dm = AmazonReviewDataModule(cfg)
dm.setup()

print(dm.summary())

{'num_users': 25077, 'num_items': 9886, 'num_train_examples': 144779, 'num_val_examples': 6858, 'num_test_examples': 6653, 'num_interactions': 209887, 'post_filter_cold_items': True, 'cold_item_reference': 'observed', 'num_items_before_cold_filter': 9979, 'num_cold_items_removed': 93, 'num_train_examples_dropped_cold': 0, 'num_val_examples_dropped_cold': 108, 'num_test_examples_dropped_cold': 236, 'num_train_history_items_removed_cold': 0, 'num_val_history_items_removed_cold': 114, 'num_test_history_items_removed_cold': 369, 'split_strategy': 'global_time', 'global_val_cutoff': 1488585600, 'global_test_cutoff': 1507248000, 'sid_len': None, 'item_vocab_size': 9887, 'post_split_filter': {'enabled': True, 'train_item_source': 'observed', 'min_history_len': 1, 'num_items_before': 9979, 'num_items_after': 9886, 'num_items_removed': 93, 'num_train_examples_before': 144779, 'num_val_examples_before': 6966, 'num_test_examples_before': 6889, 'num_train_examples_after': 144779, 'num_val_examples

In [30]:
dm.item2id

{'B0002D0HXA': 1,
 'B001EEO8IQ': 2,
 'B002474FI0': 3,
 'B00IY0BLB0': 4,
 'B00LBKFYUO': 5,
 'B0148EQYA8': 6,
 'B019DTF2OW': 7,
 'B01EY1JLJG': 8,
 'B00282WZG0': 9,
 'B0002GLDQM': 10,
 'B0002OOMU8': 11,
 'B0002D15JU': 12,
 'B004OZ0BLU': 13,
 'B007IE5QP0': 14,
 '0739079891': 15,
 'B0002E3CK4': 16,
 'B0002H03YY': 17,
 'B0002H05BA': 18,
 'B0006LOBA8': 19,
 'B0051WAJ5S': 20,
 'B0094NVV5C': 21,
 'B016W4O7BA': 22,
 'B0047JP1D6': 23,
 'B009115NQA': 24,
 'B0002D09Y2': 25,
 'B00IGUTZX4': 26,
 'B004Z17008': 27,
 'B00MXUJ394': 28,
 'B00RGNR4TE': 29,
 'B0009G1E0K': 30,
 'B000ZJTPLG': 31,
 'B0002FOZCM': 32,
 'B0073XCXHA': 33,
 'B005PNY6V6': 34,
 'B004PFWYL4': 35,
 'B0002BG2DQ': 36,
 'B00CDA0IUC': 37,
 'B0069R0U3U': 38,
 'B0037MA738': 39,
 'B000PO30QM': 40,
 'B007YWHIP2': 41,
 'B0064RTS0G': 42,
 'B010CI6M88': 43,
 'B00BL6JDUA': 44,
 'B004XNK7AI': 45,
 'B009MBTCKW': 46,
 'B00U49CFUA': 47,
 'B00YA48LJY': 48,
 'B0002GLMEK': 49,
 'B00YQXI2HU': 50,
 'B0002H0A3S': 51,
 'B0007Y09VO': 52,
 'B002026DR0': 53,
 '

In [31]:
dm.id2item

['<PAD>',
 'B0002D0HXA',
 'B001EEO8IQ',
 'B002474FI0',
 'B00IY0BLB0',
 'B00LBKFYUO',
 'B0148EQYA8',
 'B019DTF2OW',
 'B01EY1JLJG',
 'B00282WZG0',
 'B0002GLDQM',
 'B0002OOMU8',
 'B0002D15JU',
 'B004OZ0BLU',
 'B007IE5QP0',
 '0739079891',
 'B0002E3CK4',
 'B0002H03YY',
 'B0002H05BA',
 'B0006LOBA8',
 'B0051WAJ5S',
 'B0094NVV5C',
 'B016W4O7BA',
 'B0047JP1D6',
 'B009115NQA',
 'B0002D09Y2',
 'B00IGUTZX4',
 'B004Z17008',
 'B00MXUJ394',
 'B00RGNR4TE',
 'B0009G1E0K',
 'B000ZJTPLG',
 'B0002FOZCM',
 'B0073XCXHA',
 'B005PNY6V6',
 'B004PFWYL4',
 'B0002BG2DQ',
 'B00CDA0IUC',
 'B0069R0U3U',
 'B0037MA738',
 'B000PO30QM',
 'B007YWHIP2',
 'B0064RTS0G',
 'B010CI6M88',
 'B00BL6JDUA',
 'B004XNK7AI',
 'B009MBTCKW',
 'B00U49CFUA',
 'B00YA48LJY',
 'B0002GLMEK',
 'B00YQXI2HU',
 'B0002H0A3S',
 'B0007Y09VO',
 'B002026DR0',
 'B0017SZ4OG',
 'B0054QM4D6',
 'B00N4ZHPZ6',
 'B00XQFONF4',
 'B00OKA9F8O',
 'B005CXV6PI',
 'B00DMACALC',
 'B00DJU340G',
 'B00ILAQ974',
 'B0002E1NNC',
 'B0002E1NWI',
 'B0002F7IEE',
 'B0002E3B78',


In [32]:
len(dm.item2id)

9886

In [33]:
len(dm.id2item)

9887

In [34]:
metadata_path = Path("../data/raw/2018/meta_Musical_Instruments.json.gz")
output_path = Path("../artifacts/item_embeddings/musical_instruments_content_embeddings.pt")
output_path.parent.mkdir(parents=True, exist_ok=True)

In [35]:
text_builder = AmazonItemTextBuilder(
    include_title=True,
    include_categories=True,
    include_features=True,
    include_description=True,
    include_details=True,
    include_price=False,
    max_chars=4096,
)

embedder = HashingTextEmbedder(
    dim=384,
    lowercase=True,
    use_bigrams=True,
    normalize=True,
)

builder = ContentEmbeddingBuilder(
    text_builder=text_builder,
    embedder=embedder,
    item_key="parent_asin",
)

result = builder.build_from_metadata(
    metadata_path=metadata_path,
    batch_size=512,
    allowed_item_ids=list(dm.item2id.keys()),
)

result.store.save(output_path)

print("Saved embeddings to:", output_path)
print("Number of items:", len(result.store.item_ids))
print("Embedding shape:", result.store.embeddings.shape)

Saved embeddings to: ../artifacts/item_embeddings/musical_instruments_content_embeddings.pt
Number of items: 9886
Embedding shape: torch.Size([9886, 384])


In [36]:
embeddings_path = Path("../artifacts/item_embeddings/musical_instruments_content_embeddings.pt")
output_dir = Path("../artifacts/tokenizers/rqvae_musical_instruments")
output_dir.mkdir(parents=True, exist_ok=True)

In [37]:
embedding_store = ItemEmbeddingStore.load(embeddings_path)

item_ids = embedding_store.item_ids
embeddings = embedding_store.embeddings

print("Number of items:", len(item_ids))
print("Embedding shape:", embeddings.shape)

Number of items: 9886
Embedding shape: torch.Size([9886, 384])


In [38]:
rqvae_cfg = RQVAEConfig(
    input_dim=embeddings.shape[1],
    latent_dim=32,
    hidden_dims=(512, 256, 128),
    num_codebooks=3,
    codebook_size=256,
    commitment_weight=0.25,
    dropout=0.0,
    use_batch_norm=False,
)

train_cfg = RQVAETrainConfig(
    epochs=10,
    batch_size=1024,
    learning_rate=1e-3,
    weight_decay=0.0,
    val_ratio=0.05,
    seed=42,
    gradient_clip_norm=1.0,
    device="cpu",  # use "cpu" if needed
)

tokenizer_cfg = RQVAEItemTokenizerConfig(
    rqvae=rqvae_cfg,
    train=train_cfg,

    # Recommended for text embeddings.
    normalize="l2",

    # Important for TIGER-style flat token vocabulary:
    # code_l -> code_l + l * codebook_size
    level_offset_tokens=True,

    # Resolve duplicate semantic IDs by appending a suffix token.
    collision_policy="append",

    encode_batch_size=4096,
)


In [39]:
tokenizer = RQVAEItemTokenizer(tokenizer_cfg)

tokenizer.fit(
    embeddings=embeddings,
    item_ids=item_ids,
)

Initializing RQ-VAE codebooks from data...


rqvae epoch 1/10: 100%|██████████| 10/10 [00:00<00:00, 35.28it/s, loss=0.00208, recon_loss=0.00208, vq_loss=2e-6, dead_codes_l0=209, dead_codes_l1=43, dead_codes_l2=40]


{"epoch": 1, "step": 10, "train_dead_codes_l0": 223.609881, "train_dead_codes_l1": 45.07155, "train_dead_codes_l2": 52.320273, "train_loss": 0.002489, "train_recon_loss": 0.002485, "train_vq_loss": 4e-06, "val_loss": 0.002062, "val_recon_loss": 0.002059, "val_vq_loss": 3e-06}


rqvae epoch 2/10: 100%|██████████| 10/10 [00:00<00:00, 35.88it/s, loss=0.00203, recon_loss=0.00202, vq_loss=2e-6, dead_codes_l0=182, dead_codes_l1=37, dead_codes_l2=47]


{"epoch": 2, "step": 20, "train_dead_codes_l0": 186.143101, "train_dead_codes_l1": 41.143101, "train_dead_codes_l2": 49.398637, "train_loss": 0.002061, "train_recon_loss": 0.002059, "train_vq_loss": 2e-06, "val_loss": 0.00202, "val_recon_loss": 0.002018, "val_vq_loss": 2e-06}


rqvae epoch 3/10: 100%|██████████| 10/10 [00:00<00:00, 36.57it/s, loss=0.002, recon_loss=0.002, vq_loss=2e-6, dead_codes_l0=33, dead_codes_l1=29, dead_codes_l2=27]    


... (11 outputs left out)

rqvae epoch 9/10: 100%|██████████| 10/10 [00:00<00:00, 36.61it/s, loss=0.00183, recon_loss=0.00182, vq_loss=9e-6, dead_codes_l0=6, dead_codes_l1=33, dead_codes_l2=24]


{"epoch": 9, "step": 90, "train_dead_codes_l0": 11.451448, "train_dead_codes_l1": 25.69506, "train_dead_codes_l2": 30.432709, "train_loss": 0.001888, "train_recon_loss": 0.00188, "train_vq_loss": 8e-06, "val_loss": 0.001807, "val_recon_loss": 0.001797, "val_vq_loss": 1.1e-05}


rqvae epoch 10/10: 100%|██████████| 10/10 [00:00<00:00, 36.36it/s, loss=0.00179, recon_loss=0.00178, vq_loss=8e-6, dead_codes_l0=10, dead_codes_l1=21, dead_codes_l2=36]


{"epoch": 10, "step": 100, "train_dead_codes_l0": 8.364566, "train_dead_codes_l1": 26.23339, "train_dead_codes_l2": 33.383305, "train_loss": 0.00184, "train_recon_loss": 0.001831, "train_vq_loss": 9e-06, "val_loss": 0.001775, "val_recon_loss": 0.001765, "val_vq_loss": 9e-06}


In [40]:
tokenizer.save(output_dir)

semantic_ids_path = output_dir / "semantic_ids.json"

print("Saved tokenizer to:", output_dir)
print("Saved semantic IDs to:", semantic_ids_path)

Saved tokenizer to: ../artifacts/tokenizers/rqvae_musical_instruments
Saved semantic IDs to: ../artifacts/tokenizers/rqvae_musical_instruments/semantic_ids.json


In [41]:
semantic_id_store = tokenizer.semantic_id_store
semantic_id_mapping = semantic_id_store.to_plain_dict()
semantic_id_mapping

{'0739079891': [233, 365, 670, 768],
 '1480360295': [0, 473, 726, 768],
 '1928571018': [184, 438, 696, 768],
 '9792372326': [120, 425, 587, 768],
 'B00000J50W': [160, 447, 580, 768],
 'B00001W0DT': [233, 498, 640, 768],
 'B00001W0DH': [215, 375, 702, 768],
 'B00004TT3S': [182, 261, 625, 768],
 'B00004UE29': [111, 431, 622, 768],
 'B00004UE2D': [182, 300, 533, 768],
 'B00004Y2V1': [225, 355, 697, 768],
 'B00004Y2UT': [227, 261, 662, 768],
 'B00004Y2V2': [4, 473, 533, 768],
 'B00005ML71': [167, 455, 543, 768],
 'B000068NTU': [124, 424, 558, 768],
 'B000068NSX': [83, 418, 639, 768],
 'B000068NUX': [124, 256, 742, 768],
 'B000068NTC': [127, 455, 571, 768],
 'B000068NVJ': [114, 432, 691, 768],
 'B000068NSS': [83, 257, 670, 768],
 'B000068NW9': [124, 466, 536, 768],
 'B000068NZG': [214, 506, 613, 768],
 'B000068NZD': [194, 362, 621, 768],
 'B000068NYU': [124, 359, 673, 768],
 'B000068NZB': [124, 358, 622, 768],
 'B000068NYP': [194, 362, 602, 768],
 'B000068NYM': [124, 297, 706, 768],
 'B0000

In [42]:
len(semantic_id_mapping)

9886

In [43]:
dm.add_semantic_ids(semantic_ids=semantic_id_mapping)

In [44]:
dm.semantic_ids

{'0739079891': [234, 366, 671, 769],
 '1480360295': [1, 474, 727, 769],
 '1928571018': [185, 439, 697, 769],
 '9792372326': [121, 426, 588, 769],
 'B00000J50W': [161, 448, 581, 769],
 'B00001W0DT': [234, 499, 641, 769],
 'B00001W0DH': [216, 376, 703, 769],
 'B00004TT3S': [183, 262, 626, 769],
 'B00004UE29': [112, 432, 623, 769],
 'B00004UE2D': [183, 301, 534, 769],
 'B00004Y2V1': [226, 356, 698, 769],
 'B00004Y2UT': [228, 262, 663, 769],
 'B00004Y2V2': [5, 474, 534, 769],
 'B00005ML71': [168, 456, 544, 769],
 'B000068NTU': [125, 425, 559, 769],
 'B000068NSX': [84, 419, 640, 769],
 'B000068NUX': [125, 257, 743, 769],
 'B000068NTC': [128, 456, 572, 769],
 'B000068NVJ': [115, 433, 692, 769],
 'B000068NSS': [84, 258, 671, 769],
 'B000068NW9': [125, 467, 537, 769],
 'B000068NZG': [215, 507, 614, 769],
 'B000068NZD': [195, 363, 622, 769],
 'B000068NYU': [125, 360, 674, 769],
 'B000068NZB': [125, 359, 623, 769],
 'B000068NYP': [195, 363, 603, 769],
 'B000068NYM': [125, 298, 707, 769],
 'B0000

In [45]:
len(dm.semantic_ids)

9886

In [46]:
for batch_idx, raw_batch in enumerate(dm.train_dataloader()):
    print(f"Batch {batch_idx}:")
    print(raw_batch)
    break

Batch 0:
{'user_ids': tensor([11605, 16036,  7137, 10539,  1545, 11406, 11739, 11262, 22029,   136,
         3479, 23016, 18445, 11508,  3487, 14013, 16613,  5538, 15342, 14737,
          148, 14281, 15338,  3857,  3878,  1640, 21672,  1889, 19696, 13949,
         7707, 24374, 14452, 13879, 22486,  5790, 13518, 16649, 17273,  6078,
         5705, 16218,  3065, 14534, 13922,  8628, 21529, 16579,  9441,   765,
         9482, 24403, 11848, 20415, 21979,  1540,  9742, 16285,  8353,   187,
        17713, 24885, 12583, 14139, 16825, 12169,  3230,   977, 21506,  1439,
        22247, 20038, 22451,  1195, 12362, 20309,  4836,  4478,  5193, 18831,
        19991,  2060, 15564, 21680,  4579, 20421,  8201,  7411,  2022,  1419,
        19731,  8719, 23765, 14408,  1067,  4715,  8729, 10827, 19620,   705,
        15502, 11394,  9907, 15795, 15742, 21168,  1611, 23434,  8106,  6265,
         3113, 11784,  8614, 23270, 16048,  9079, 20094, 21795, 16777, 24571,
         7879, 15236, 17228,  2886,  7246,